In [ ]:
import copy
import importlib
import json
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
from sklearn.metrics import auc, confusion_matrix, roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import modules

importlib.reload(modules)

from modules import *
from utils import *


DTYPE = torch.float32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)


# ============================================================
# 固定随机种子
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 轻量级一维 CNN 分类器
#
# 输入:  [batch_size, num_features]
# 输出:  [batch_size, num_class]
#
# 为兼容原 MultiView.forward()，后续仍赋值给
# model.mlp_classifier，但内部实际已经是 CNN。
# ============================================================
class VOC1DCNNClassifier(nn.Module):
    def __init__(
        self,
        num_features,
        num_class=2,
        channels=(16, 32),
        pooled_length=8,
        dropout=0.30,
    ):
        super().__init__()

        self.num_features = int(num_features)
        self.num_class = int(num_class)
        self.channels = tuple(channels)
        self.pooled_length = int(pooled_length)
        self.dropout = float(dropout)

        channel_1, channel_2 = self.channels

        if channel_1 % 4 != 0:
            raise ValueError("channels[0] 必须能被 4 整除，以便使用 GroupNorm。")
        if channel_2 % 8 != 0:
            raise ValueError("channels[1] 必须能被 8 整除，以便使用 GroupNorm。")

        self.feature_extractor = nn.Sequential(
            # [B, 1, F] -> [B, C1, F]
            nn.Conv1d(
                in_channels=1,
                out_channels=channel_1,
                kernel_size=7,
                padding=3,
                bias=False,
            ),
            nn.GroupNorm(
                num_groups=4,
                num_channels=channel_1,
            ),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            # [B, C1, F/2] -> [B, C2, F/2]
            nn.Conv1d(
                in_channels=channel_1,
                out_channels=channel_2,
                kernel_size=5,
                padding=2,
                bias=False,
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=channel_2,
            ),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            # 固定输出长度，避免大规模全连接层
            nn.AdaptiveAvgPool1d(output_size=self.pooled_length),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(self.dropout),
            nn.Linear(channel_2 * self.pooled_length, 32),
            nn.GELU(),
            nn.Dropout(self.dropout / 2),
            nn.Linear(32, self.num_class),
        )

    def forward(self, x):
        if x.ndim != 2:
            raise ValueError(
                "VOC1DCNNClassifier 输入应为二维张量 "
                f"[batch, features]，当前形状为 {tuple(x.shape)}"
            )

        if x.shape[1] != self.num_features:
            raise ValueError(
                f"期望 {self.num_features} 个特征，"
                f"实际输入 {x.shape[1]} 个特征。"
            )

        # [B, F] -> [B, 1, F]
        x = x.unsqueeze(1)
        x = self.feature_extractor(x)
        return self.classifier(x)


def count_trainable_parameters(module):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


# ============================================================
# 计算分类指标
# ============================================================
def compute_metrics(cm, targets, probs):
    tn = float(cm[0, 0])
    fp = float(cm[0, 1])
    fn = float(cm[1, 0])
    tp = float(cm[1, 1])

    eps = 1e-12

    sensitivity = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    ppv = tp / (tp + fp + eps)
    npv = tn / (tn + fn + eps)
    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    f1 = 2 * ppv * sensitivity / (ppv + sensitivity + eps)

    try:
        auc_value = roc_auc_score(targets, probs)
    except ValueError:
        auc_value = np.nan

    return {
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "PPV": ppv,
        "NPV": npv,
        "Accuracy": accuracy,
        "F1": f1,
        "AUC": auc_value,
    }


# ============================================================
# 均值、标准差、标准误和 95% 置信区间
# ============================================================
def mean_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    n = len(values)

    if n == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    mean_value = float(values.mean())

    if n < 2:
        return mean_value, np.nan, np.nan, np.nan, np.nan

    std_value = float(values.std(ddof=1))
    sem_value = std_value / np.sqrt(n)
    t_critical = stats.t.ppf(1 - alpha / 2, df=n - 1)
    ci_lower = mean_value - t_critical * sem_value
    ci_upper = mean_value + t_critical * sem_value

    return mean_value, std_value, sem_value, ci_lower, ci_upper



# ============================================================
# 构建 CNN-MultiView 模型
# ============================================================
def build_cnn_multiview(
    k_means_mask,
    num_features,
    temp_start,
    cnn_channels,
    cnn_pooled_length,
    cnn_dropout,
):
    model = MultiView(
        k_mean_mask=k_means_mask,
        num_blocks=4,
        head_dim=256,
        num_class=2,
        temperature=temp_start,
    )

    original_mlp_parameter_count = count_trainable_parameters(
        model.mlp_classifier
    )

    model.mlp_classifier = VOC1DCNNClassifier(
        num_features=num_features,
        num_class=2,
        channels=cnn_channels,
        pooled_length=cnn_pooled_length,
        dropout=cnn_dropout,
    )

    cnn_parameter_count = count_trainable_parameters(
        model.mlp_classifier
    )

    model = model.to(DEVICE, dtype=DTYPE)

    return (
        model,
        original_mlp_parameter_count,
        cnn_parameter_count,
    )


# ============================================================
# 在验证集上评价一个已经确定的冠军模型
# ============================================================
def evaluate_cnn_on_validation(
    model,
    val_loader,
    x_val,
    binary_mask,
    saved_soft_score=None,
):
    model.eval()

    with torch.no_grad():
        final_mask = binary_mask.to(DEVICE, dtype=DTYPE)

        if saved_soft_score is None:
            soft_score = (
                model.selection_prob(
                    x_val.to(DEVICE, dtype=DTYPE)
                )
                .detach()
                .cpu()
                .numpy()
            )
        else:
            if torch.is_tensor(saved_soft_score):
                soft_score = (
                    saved_soft_score
                    .detach()
                    .cpu()
                    .numpy()
                )
            else:
                soft_score = np.asarray(saved_soft_score)

        group_predictions = []
        group_probabilities = []
        group_targets = []

        for x_eval, y_eval in val_loader:
            x_eval = x_eval.to(DEVICE, dtype=DTYPE)
            y_eval = y_eval.to(DEVICE).long()

            outputs = model.mlp_classifier(
                x_eval * final_mask
            )
            probabilities = torch.softmax(
                outputs.float(), dim=1
            )[:, 1]
            predictions = outputs.argmax(dim=1)

            group_predictions.extend(
                predictions.detach().cpu().numpy()
            )
            group_probabilities.extend(
                probabilities.detach().cpu().numpy()
            )
            group_targets.extend(
                y_eval.detach().cpu().numpy()
            )

    group_predictions = np.asarray(group_predictions)
    group_probabilities = np.asarray(group_probabilities)
    group_targets = np.asarray(group_targets)

    group_cm = confusion_matrix(
        group_targets,
        group_predictions,
        labels=[0, 1],
    )

    return (
        soft_score,
        group_predictions,
        group_probabilities,
        group_targets,
        group_cm,
    )


# ============================================================
# 检查旧 checkpoint 是否与当前续跑配置兼容
# ============================================================
def validate_resume_checkpoint(
    checkpoint,
    checkpoint_path,
    expected_base_seed,
    expected_split_seed,
    expected_num_features,
    expected_channels,
    expected_pooled_length,
    expected_dropout,
):
    classifier_type = checkpoint.get(
        "classifier_type",
        "VOC1DCNNClassifier",
    )
    if classifier_type != "VOC1DCNNClassifier":
        raise ValueError(
            f"{checkpoint_path} 的分类器不是 VOC1DCNNClassifier。"
        )

    if "base_seed" in checkpoint:
        if int(checkpoint["base_seed"]) != int(expected_base_seed):
            raise ValueError(
                f"{checkpoint_path} 的 base_seed 与当前 seed 不一致。"
            )

    if "split_seed" in checkpoint:
        if int(checkpoint["split_seed"]) != int(expected_split_seed):
            raise ValueError(
                f"{checkpoint_path} 的 split_seed 与当前分组不一致。"
            )

    config = checkpoint.get("classifier_config", {})

    if config:
        if int(config.get("num_features", expected_num_features)) != int(
            expected_num_features
        ):
            raise ValueError(
                f"{checkpoint_path} 的特征数与当前数据不一致。"
            )

        saved_channels = tuple(
            config.get("channels", expected_channels)
        )
        if saved_channels != tuple(expected_channels):
            raise ValueError(
                f"{checkpoint_path} 的 CNN channels={saved_channels}，"
                f"当前为 {tuple(expected_channels)}。不能混合续跑。"
            )

        saved_pooled_length = int(
            config.get("pooled_length", expected_pooled_length)
        )
        if saved_pooled_length != int(expected_pooled_length):
            raise ValueError(
                f"{checkpoint_path} 的 pooled_length 不一致。"
            )

        saved_dropout = float(
            config.get("dropout", expected_dropout)
        )
        if not np.isclose(saved_dropout, float(expected_dropout)):
            raise ValueError(
                f"{checkpoint_path} 的 dropout 不一致。"
            )


# ============================================================
# 每完成一组就保存断点状态，防止再次断连
# ============================================================
def save_resume_progress(
    out_dir,
    all_metrics,
    experiment_repeats,
    status="running",
):
    if not all_metrics:
        return

    partial_df = (
        pd.DataFrame(all_metrics)
        .sort_values("Repeat")
        .reset_index(drop=True)
    )

    partial_path = os.path.join(
        out_dir,
        "resume_partial_val_metrics.csv",
    )
    partial_tmp = partial_path + ".tmp"
    partial_df.to_csv(partial_tmp, index=False)
    os.replace(partial_tmp, partial_path)

    completed_repeats = [
        int(value)
        for value in partial_df["Repeat"].tolist()
    ]

    status_data = {
        "status": status,
        "completed_count": len(completed_repeats),
        "target_count": int(experiment_repeats),
        "completed_repeats": completed_repeats,
        "last_updated": pd.Timestamp.now().isoformat(),
    }

    status_path = os.path.join(
        out_dir,
        "resume_status.json",
    )
    status_tmp = status_path + ".tmp"
    with open(status_tmp, "w", encoding="utf-8") as file:
        json.dump(status_data, file, indent=2, ensure_ascii=False)
    os.replace(status_tmp, status_path)


# ============================================================
# 支持断点续跑的 CNN 主实验
#
# resume=True 时：
# 1. 自动扫描 champion_model_repeat*.pt
# 2. 已存在的组只加载并重新计算验证集指标
# 3. 缺失的组继续训练
# 4. 最后合并为完整 experiment_repeats 组结果
# ============================================================
def experiment(
    data_path,
    save_path,
    num_cluster,
    experiment_repeats=2,
    repeats=2,
    epochs=200,
    sparsity_lambda=1e-2,
    temp_start=1.0,
    temp_end=0.3,
    seed=42,
    panel_threshold=0.5,
    cnn_channels=(16, 32),
    cnn_pooled_length=8,
    cnn_dropout=0.30,
    resume=True,
):
    set_seed(seed)

    out_dir = os.path.dirname(save_path)
    if not out_dir:
        raise ValueError(
            "save_path 必须包含结果目录，例如 "
            "./result/unknown_val_cnn_seed_selection/all_masks.pt"
        )

    os.makedirs(out_dir, exist_ok=True)

    data = sio.loadmat(data_path)
    samples = torch.tensor(data["X"], dtype=DTYPE)
    labels = torch.tensor(data["y"], dtype=DTYPE).view(-1).long()
    voc_names = [
        str(value.flat[0])
        for value in data["feat_names"].flatten()
    ]

    print(f"样本数: {samples.shape[0]}")
    print(f"特征数: {samples.shape[1]}")
    print(f"类别数量: {torch.bincount(labels).tolist()}")

    # 必须与第一次运行保持完全相同：先固定 base seed，再做外层划分
    train_pool_loader, sealed_test_loader, _ = split_dataset(
        samples,
        labels,
        batch_size=16,
        split_length=[0.8, 0.2],
    )

    temporary_loader = DataLoader(
        train_pool_loader.dataset,
        batch_size=len(train_pool_loader.dataset),
        shuffle=False,
    )
    x_pool, y_pool = next(iter(temporary_loader))

    _ = sealed_test_loader
    print("测试集已封存，本阶段不参与模型和随机种子筛选。")

    # 必须与原实验在相同随机状态下生成
    k_means_mask = feature_cluster(
        x_pool.float().numpy(),
        num_cluster,
    )

    existing_indices = []
    for out_idx in range(experiment_repeats):
        checkpoint_path = os.path.join(
            out_dir,
            f"champion_model_repeat{out_idx}.pt",
        )
        if os.path.isfile(checkpoint_path):
            existing_indices.append(out_idx)

    if existing_indices and not resume:
        raise RuntimeError(
            f"检测到 {len(existing_indices)} 个旧 checkpoint。"
            "若要续跑，请设置 resume=True；"
            "若要重跑，请使用新的结果目录。"
        )

    print(
        f"检测到已保存 checkpoint: {len(existing_indices)}/"
        f"{experiment_repeats}"
    )
    if existing_indices:
        print(
            "已保存 repeat 索引范围: "
            f"{min(existing_indices)} 到 {max(existing_indices)}"
        )
        print("将加载已有组，只训练缺失组。")

    all_masks = []
    all_soft = []
    all_cm = []
    all_probs = []
    all_targets = []
    all_metrics = []

    original_mlp_parameter_count = None
    cnn_parameter_count = None

    progress_bar = tqdm(
        range(experiment_repeats),
        desc="CNN resume",
    )

    for out_idx in progress_bar:
        split_seed = seed + 100000 + out_idx
        set_seed(split_seed)

        train_loader_init, val_loader, _ = split_dataset(
            x_pool,
            y_pool,
            batch_size=16,
            split_length=[0.75, 0.25],
        )

        train_loader_init = DataLoader(
            train_loader_init.dataset,
            batch_size=16,
            shuffle=True,
            drop_last=True,
        )

        x_val = torch.cat([x for x, _ in val_loader])

        checkpoint_path = os.path.join(
            out_dir,
            f"champion_model_repeat{out_idx}.pt",
        )

        # ====================================================
        # A. 已存在 checkpoint：加载，不重复训练
        # ====================================================
        if resume and os.path.isfile(checkpoint_path):
            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
                weights_only=False,
            )

            validate_resume_checkpoint(
                checkpoint=checkpoint,
                checkpoint_path=checkpoint_path,
                expected_base_seed=seed,
                expected_split_seed=split_seed,
                expected_num_features=samples.shape[1],
                expected_channels=cnn_channels,
                expected_pooled_length=cnn_pooled_length,
                expected_dropout=cnn_dropout,
            )

            model_seed = int(
                checkpoint.get(
                    "model_seed",
                    seed + out_idx * repeats,
                )
            )
            set_seed(model_seed)

            (
                model,
                current_mlp_count,
                current_cnn_count,
            ) = build_cnn_multiview(
                k_means_mask=k_means_mask,
                num_features=samples.shape[1],
                temp_start=temp_start,
                cnn_channels=cnn_channels,
                cnn_pooled_length=cnn_pooled_length,
                cnn_dropout=cnn_dropout,
            )

            if original_mlp_parameter_count is None:
                original_mlp_parameter_count = current_mlp_count
                cnn_parameter_count = current_cnn_count

            model.load_state_dict(
                checkpoint["state_dict"],
                strict=True,
            )

            binary_mask = (
                checkpoint["binary_mask"]
                .detach()
                .cpu()
            )

            (
                soft_score,
                _,
                group_probabilities,
                group_targets,
                group_cm,
            ) = evaluate_cnn_on_validation(
                model=model,
                val_loader=val_loader,
                x_val=x_val,
                binary_mask=binary_mask,
                saved_soft_score=checkpoint.get("soft_score"),
            )

            metrics = compute_metrics(
                group_cm,
                group_targets,
                group_probabilities,
            )
            metrics["Repeat"] = out_idx
            metrics["Split_Seed"] = int(
                checkpoint.get("split_seed", split_seed)
            )
            metrics["Model_Seed"] = model_seed
            metrics["Inner_Index"] = int(
                checkpoint.get("inner_index", -1)
            )
            metrics["Champion_Val_Acc"] = float(
                checkpoint.get("val_acc", metrics["Accuracy"])
            )

            all_metrics.append(metrics)
            all_masks.append(binary_mask)
            all_soft.append(soft_score)
            all_cm.append(
                torch.from_numpy(group_cm).unsqueeze(0).float()
            )
            all_probs.append(group_probabilities)
            all_targets.append(group_targets)

            progress_bar.set_postfix(
                {
                    "mode": "loaded",
                    "repeat": out_idx,
                    "done": len(all_metrics),
                }
            )

            save_resume_progress(
                out_dir=out_dir,
                all_metrics=all_metrics,
                experiment_repeats=experiment_repeats,
                status="running",
            )
            continue

        # ====================================================
        # B. checkpoint 不存在：训练这一组
        # ====================================================
        group_best_acc = -1.0
        group_best_mask = None
        group_best_wts = None
        group_best_seed = None
        group_best_in_idx = None

        for in_idx in range(repeats):
            run_seed = seed + out_idx * repeats + in_idx
            set_seed(run_seed)

            (
                model,
                current_mlp_count,
                current_cnn_count,
            ) = build_cnn_multiview(
                k_means_mask=k_means_mask,
                num_features=samples.shape[1],
                temp_start=temp_start,
                cnn_channels=cnn_channels,
                cnn_pooled_length=cnn_pooled_length,
                cnn_dropout=cnn_dropout,
            )

            if original_mlp_parameter_count is None:
                original_mlp_parameter_count = current_mlp_count
                cnn_parameter_count = current_cnn_count

                parameter_change = (
                    cnn_parameter_count
                    / max(original_mlp_parameter_count, 1)
                    - 1.0
                ) * 100.0

                print("=" * 60)
                print("Classifier replacement: MLP -> 1D CNN")
                print(
                    f"Original MLP parameters: "
                    f"{original_mlp_parameter_count:,}"
                )
                print(
                    f"New CNN parameters:      "
                    f"{cnn_parameter_count:,}"
                )
                print(
                    f"Parameter change:         "
                    f"{parameter_change:+.2f}%"
                )
                print("=" * 60)

            optimizer = optim.Adam(
                [
                    {
                        "params": model.logist.parameters(),
                        "lr": 1e-5,
                        "weight_decay": 1e-4,
                    },
                    {
                        "params": model.mlp_classifier.parameters(),
                        "lr": 5e-4,
                        "weight_decay": 5e-3,
                    },
                ]
            )
            criterion = nn.CrossEntropyLoss()

            local_best_acc = -1.0
            local_best_wts = None
            local_best_mask = None

            for epoch in range(epochs):
                model.temperature = (
                    temp_start
                    + (temp_end - temp_start)
                    * (epoch / max(1, epochs - 1))
                )

                model.train()
                for x_batch, y_batch in train_loader_init:
                    x_batch = x_batch.to(DEVICE, dtype=DTYPE)
                    y_batch = y_batch.to(DEVICE).long()

                    optimizer.zero_grad()
                    outputs, _ = model(x_batch)
                    sparsity = model.selection_prob(x_batch).mean()
                    loss = (
                        criterion(outputs, y_batch)
                        + sparsity_lambda * sparsity
                    )
                    loss.backward()
                    optimizer.step()

                model.eval()
                correct = 0
                total_number = 0

                with torch.no_grad():
                    mask_eval = model.get_score(
                        x_val.to(DEVICE, dtype=DTYPE)
                    )

                    for x_batch, y_batch in val_loader:
                        x_batch = x_batch.to(DEVICE, dtype=DTYPE)
                        y_batch = y_batch.to(DEVICE).long()

                        outputs = model.mlp_classifier(
                            x_batch * mask_eval
                        )
                        predictions = outputs.argmax(dim=1)
                        correct += (
                            predictions == y_batch
                        ).sum().item()
                        total_number += y_batch.size(0)

                    validation_accuracy = (
                        correct / max(total_number, 1)
                    )

                if validation_accuracy > local_best_acc:
                    local_best_acc = validation_accuracy
                    local_best_wts = copy.deepcopy(
                        model.state_dict()
                    )
                    local_best_mask = (
                        mask_eval.detach().cpu()
                    )

            if local_best_acc > group_best_acc:
                group_best_acc = local_best_acc
                group_best_wts = local_best_wts
                group_best_mask = local_best_mask
                group_best_seed = run_seed
                group_best_in_idx = in_idx

        if group_best_wts is None:
            raise RuntimeError(
                f"第 {out_idx} 组没有得到有效冠军模型。"
            )

        set_seed(group_best_seed)
        (
            champion_model,
            current_mlp_count,
            current_cnn_count,
        ) = build_cnn_multiview(
            k_means_mask=k_means_mask,
            num_features=samples.shape[1],
            temp_start=temp_start,
            cnn_channels=cnn_channels,
            cnn_pooled_length=cnn_pooled_length,
            cnn_dropout=cnn_dropout,
        )
        champion_model.load_state_dict(
            group_best_wts,
            strict=True,
        )

        (
            soft_score,
            _,
            group_probabilities,
            group_targets,
            group_cm,
        ) = evaluate_cnn_on_validation(
            model=champion_model,
            val_loader=val_loader,
            x_val=x_val,
            binary_mask=group_best_mask,
            saved_soft_score=None,
        )

        metrics = compute_metrics(
            group_cm,
            group_targets,
            group_probabilities,
        )
        metrics["Repeat"] = out_idx
        metrics["Split_Seed"] = split_seed
        metrics["Model_Seed"] = group_best_seed
        metrics["Inner_Index"] = group_best_in_idx
        metrics["Champion_Val_Acc"] = group_best_acc

        checkpoint = {
            "state_dict": group_best_wts,
            "binary_mask": group_best_mask,
            "soft_score": torch.from_numpy(soft_score),
            "val_acc": group_best_acc,
            "base_seed": seed,
            "split_seed": split_seed,
            "model_seed": group_best_seed,
            "inner_index": group_best_in_idx,
            "evaluation_set": "val",
            "classifier_type": "VOC1DCNNClassifier",
            "classifier_config": {
                "num_features": int(samples.shape[1]),
                "channels": list(cnn_channels),
                "kernel_sizes": [7, 5],
                "pooled_length": int(cnn_pooled_length),
                "dropout": float(cnn_dropout),
                "num_class": 2,
            },
            "training_config": {
                "num_cluster": int(num_cluster),
                "experiment_repeats": int(experiment_repeats),
                "repeats": int(repeats),
                "epochs": int(epochs),
                "sparsity_lambda": float(sparsity_lambda),
                "temp_start": float(temp_start),
                "temp_end": float(temp_end),
                "panel_threshold": float(panel_threshold),
            },
            "original_mlp_parameter_count": (
                original_mlp_parameter_count
            ),
            "cnn_parameter_count": cnn_parameter_count,
        }

        torch.save(checkpoint, checkpoint_path)

        all_metrics.append(metrics)
        all_masks.append(group_best_mask)
        all_soft.append(soft_score)
        all_cm.append(
            torch.from_numpy(group_cm).unsqueeze(0).float()
        )
        all_probs.append(group_probabilities)
        all_targets.append(group_targets)

        progress_bar.set_postfix(
            {
                "mode": "trained",
                "repeat": out_idx,
                "val_acc": f"{group_best_acc:.3f}",
                "done": len(all_metrics),
            }
        )

        save_resume_progress(
            out_dir=out_dir,
            all_metrics=all_metrics,
            experiment_repeats=experiment_repeats,
            status="running",
        )

    if len(all_metrics) != experiment_repeats:
        raise RuntimeError(
            f"最终只汇总到 {len(all_metrics)} 组，"
            f"目标是 {experiment_repeats} 组。"
        )

    print(
        f"[完成] 已汇总 {len(all_metrics)} 个 CNN 冠军模型。"
    )

    # ========================================================
    # 汇总特征选择结果
    # ========================================================
    all_masks_tensor = torch.stack(all_masks)
    all_soft_array = np.stack(all_soft)
    number_of_champions = len(all_masks)

    selection_freq = (
        all_masks_tensor.float().mean(dim=0).cpu().numpy()
    )
    soft_mean = all_soft_array.mean(axis=0)
    soft_std = all_soft_array.std(axis=0)
    soft_sem = soft_std / np.sqrt(number_of_champions)
    combined_cm = torch.cat(all_cm, dim=0)

    sorted_indices = np.lexsort(
        (soft_mean, selection_freq)
    )[::-1]
    top_n = int(
        (selection_freq > panel_threshold).sum()
    )

    if top_n == 0:
        top_n = min(20, len(soft_mean))
        sorted_indices = np.argsort(soft_mean)[::-1]

    top_indices = sorted_indices[:top_n]

    torch.save(all_masks_tensor, save_path)
    print(f"[保存1] 原始二值 mask -> {save_path}")

    config = {
        "data_path": data_path,
        "save_path": save_path,
        "seed": seed,
        "num_cluster": num_cluster,
        "experiment_repeats": experiment_repeats,
        "repeats": repeats,
        "epochs": epochs,
        "batch_size": 16,
        "split_outer": [0.8, 0.2],
        "split_inner": [0.75, 0.25],
        "num_blocks": 4,
        "head_dim": 256,
        "classifier_type": "VOC1DCNNClassifier",
        "cnn_channels": list(cnn_channels),
        "cnn_kernel_sizes": [7, 5],
        "cnn_pooled_length": int(cnn_pooled_length),
        "cnn_dropout": float(cnn_dropout),
        "original_mlp_parameter_count": (
            original_mlp_parameter_count
        ),
        "cnn_parameter_count": cnn_parameter_count,
        "lr_logist": 1e-5,
        "lr_classifier": 5e-4,
        "wd_logist": 1e-4,
        "wd_classifier": 5e-3,
        "sparsity_lambda": sparsity_lambda,
        "temp_start": temp_start,
        "temp_end": temp_end,
        "panel_threshold": panel_threshold,
        "evaluation_set": "val",
        "sealed_test_set_used": False,
        "resume_enabled": bool(resume),
        "loaded_checkpoint_count_at_start": len(existing_indices),
        "split_seed_strategy": "seed + 100000 + out_idx",
        "model_seed_strategy": (
            "seed + out_idx * repeats + in_idx"
        ),
        "torch_version": torch.__version__,
        "numpy_version": np.__version__,
    }

    config_path = os.path.join(out_dir, "run_config.json")
    with open(config_path, "w", encoding="utf-8") as file:
        json.dump(config, file, indent=2, ensure_ascii=False)
    print(f"[保存2] 运行配置 -> {config_path}")

    feature_stats_path = os.path.join(
        out_dir,
        "feature_selection_stats.csv",
    )
    feature_stats_df = pd.DataFrame(
        {
            "VOC_Index": np.arange(len(soft_mean)),
            "VOC_Name": voc_names,
            "Selection_Freq": selection_freq,
            "Soft_Mean": soft_mean,
            "Soft_Std": soft_std,
            "Soft_SEM": soft_sem,
        }
    )
    feature_stats_df.to_csv(
        feature_stats_path,
        index=False,
    )
    print(f"[保存3] 特征选择统计 -> {feature_stats_path}")

    selected_mask = selection_freq > panel_threshold
    selected_indices = np.where(selected_mask)[0]
    selected_panel_df = pd.DataFrame(
        {
            "VOC_Index": selected_indices,
            "VOC_Name": [
                voc_names[index]
                for index in selected_indices
            ],
            "Selection_Freq": selection_freq[selected_mask],
            "Soft_Mean": soft_mean[selected_mask],
            "Soft_Std": soft_std[selected_mask],
        }
    )
    selected_panel_df = (
        selected_panel_df
        .sort_values(
            ["Selection_Freq", "Soft_Mean"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    selected_panel_path = os.path.join(
        out_dir,
        "selected_voc_panel.csv",
    )
    selected_panel_df.to_csv(
        selected_panel_path,
        index=False,
    )
    print(
        f"[保存4] 入选 VOC panel 共 "
        f"{len(selected_panel_df)} 个 -> {selected_panel_path}"
    )

    cm_mean = combined_cm.mean(dim=0).numpy()
    cm_path = os.path.join(
        out_dir,
        "val_confusion_matrix_summary.csv",
    )
    pd.DataFrame(
        cm_mean,
        index=["True_Class_0", "True_Class_1"],
        columns=["Pred_Class_0", "Pred_Class_1"],
    ).to_csv(cm_path)
    print(f"[保存5] 验证集混淆矩阵均值 -> {cm_path}")

    metric_keys = [
        "Sensitivity",
        "Specificity",
        "PPV",
        "NPV",
        "Accuracy",
        "F1",
        "AUC",
    ]
    record_keys = [
        "Repeat",
        "Split_Seed",
        "Model_Seed",
        "Inner_Index",
        "Champion_Val_Acc",
    ]

    per_repeat_df = (
        pd.DataFrame(all_metrics)
        .sort_values("Repeat")
        .reset_index(drop=True)
    )[record_keys + metric_keys]

    per_repeat_path = os.path.join(
        out_dir,
        "val_metrics_per_repeat.csv",
    )
    per_repeat_df.to_csv(per_repeat_path, index=False)
    print(f"[保存6a] 验证集逐组指标 -> {per_repeat_path}")

    seed_summary_df = per_repeat_df[
        [
            "Repeat",
            "Split_Seed",
            "Model_Seed",
            "Inner_Index",
            "Champion_Val_Acc",
            "Accuracy",
            "F1",
            "AUC",
        ]
    ].copy()
    seed_summary_df = (
        seed_summary_df
        .sort_values(
            ["Champion_Val_Acc", "AUC", "F1"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    seed_summary_path = os.path.join(
        out_dir,
        "champion_seed_summary.csv",
    )
    seed_summary_df.to_csv(seed_summary_path, index=False)
    print(
        f"[保存6b] CNN 冠军模型随机种子 -> "
        f"{seed_summary_path}"
    )

    summary_rows = []
    for metric_name in metric_keys:
        (
            mean_value,
            std_value,
            sem_value,
            ci_lower,
            ci_upper,
        ) = mean_ci(per_repeat_df[metric_name].values)

        summary_rows.append(
            {
                "Metric": metric_name,
                "Mean": mean_value,
                "Std": std_value,
                "SEM": sem_value,
                "CI95_Lower": ci_lower,
                "CI95_Upper": ci_upper,
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(
        out_dir,
        "val_metrics_summary.csv",
    )
    summary_df.to_csv(summary_path, index=False)
    print(f"[保存6c] 验证集指标汇总 -> {summary_path}")

    raw_data_path = os.path.join(
        out_dir,
        "val_raw_probs_targets.npz",
    )
    np.savez(
        raw_data_path,
        probs=np.array(all_probs, dtype=object),
        targets=np.array(all_targets, dtype=object),
    )
    print(f"[保存7] 验证集原始概率和标签 -> {raw_data_path}")

    mean_fpr = np.linspace(0, 1, 200)
    tprs = []
    auc_values = []

    for probabilities, targets in zip(all_probs, all_targets):
        if len(np.unique(targets)) < 2:
            continue

        fpr, tpr, _ = roc_curve(targets, probabilities)
        auc_value = auc(fpr, tpr)
        interpolated_tpr = np.interp(mean_fpr, fpr, tpr)
        interpolated_tpr[0] = 0.0

        tprs.append(interpolated_tpr)
        auc_values.append(auc_value)

    if len(tprs) == 0:
        raise RuntimeError(
            "没有可用于绘制 ROC 的有效验证集。"
        )

    tprs = np.asarray(tprs)
    auc_values = np.asarray(auc_values)
    mean_tpr = tprs.mean(axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = tprs.std(axis=0)
    mean_auc = float(np.mean(auc_values))
    std_auc = float(np.std(auc_values))

    plot_data_path = os.path.join(
        out_dir,
        "val_plot_data.npz",
    )
    np.savez(
        plot_data_path,
        voc_names=np.array(voc_names, dtype=object),
        selection_freq=selection_freq,
        soft_mean=soft_mean,
        soft_std=soft_std,
        soft_sem=soft_sem,
        top_idx=top_indices,
        top_n=top_n,
        mean_fpr=mean_fpr,
        tprs=tprs,
        mean_tpr=mean_tpr,
        std_tpr=std_tpr,
        aucs=auc_values,
        mean_auc=mean_auc,
        std_auc=std_auc,
        cm_mean=cm_mean,
    )
    print(f"[保存8] 验证集画图数据 -> {plot_data_path}")

    auc_path = os.path.join(
        out_dir,
        "val_auc_per_group.csv",
    )
    pd.DataFrame(
        {
            "Group": np.arange(len(auc_values)),
            "AUC": auc_values,
        }
    ).to_csv(auc_path, index=False)
    print(f"[保存9] 验证集逐组 AUC -> {auc_path}")

    figure_feature, axes = plt.subplots(
        2,
        1,
        figsize=(16, 8),
    )

    axes[0].bar(
        range(top_n),
        selection_freq[top_indices],
        color="teal",
        alpha=0.7,
    )
    axes[0].axhline(
        y=panel_threshold,
        color="red",
        linestyle="--",
        label=f"Threshold {panel_threshold}",
    )
    axes[0].set_title(
        f"Top {top_n} VOC Selection Frequency "
        f"(n={number_of_champions} CNN groups)"
    )
    axes[0].set_ylabel("Selection Frequency")
    axes[0].set_ylim(0, 1.05)
    axes[0].legend()

    plot_mean = soft_mean[top_indices]
    plot_sem = soft_sem[top_indices]
    plot_std = soft_std[top_indices]

    axes[1].bar(
        range(top_n),
        plot_mean,
        color="coral",
        alpha=0.6,
        label="Soft Importance sigma(theta)",
    )
    axes[1].fill_between(
        range(top_n),
        plot_mean - plot_sem,
        plot_mean + plot_sem,
        color="gray",
        alpha=0.4,
        label="+/- SEM",
    )
    axes[1].errorbar(
        range(top_n),
        plot_mean,
        yerr=plot_std,
        fmt="none",
        ecolor="black",
        capsize=2,
        alpha=0.5,
        label="Std Dev",
    )
    axes[1].set_title(
        f"Soft Importance of Top {top_n} Features "
        f"(Ordered by Selection Freq, CNN)"
    )
    axes[1].set_ylabel("sigma(theta)")
    axes[1].set_xlabel("VOC Index (Ranked)")
    axes[1].legend()

    figure_feature.tight_layout()
    feature_figure_path = os.path.join(
        out_dir,
        "feature_importance.png",
    )
    figure_feature.savefig(
        feature_figure_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    print(f"[保存10] 特征重要性图 -> {feature_figure_path}")

    figure_roc, axis = plt.subplots(figsize=(7, 7))

    for tpr_value in tprs:
        axis.plot(
            mean_fpr,
            tpr_value,
            color="steelblue",
            alpha=0.15,
            linewidth=0.8,
        )

    axis.plot(
        mean_fpr,
        mean_tpr,
        color="navy",
        linewidth=2,
        label=(
            f"Mean ROC (AUC = {mean_auc:.3f} "
            f"+/- {std_auc:.3f})"
        ),
    )
    axis.fill_between(
        mean_fpr,
        np.maximum(mean_tpr - std_tpr, 0),
        np.minimum(mean_tpr + std_tpr, 1),
        color="steelblue",
        alpha=0.2,
        label="+/- 1 SD",
    )
    axis.plot(
        [0, 1],
        [0, 1],
        "k--",
        linewidth=1,
        label="Chance",
    )
    axis.set_xlabel("False Positive Rate", fontsize=13)
    axis.set_ylabel("True Positive Rate", fontsize=13)
    axis.set_title(
        f"ROC Curve (n={number_of_champions} "
        f"CNN champion models, VAL set)",
        fontsize=14,
    )
    axis.legend(loc="lower right", fontsize=11)
    axis.set_xlim([0, 1])
    axis.set_ylim([0, 1.02])

    figure_roc.tight_layout()
    roc_path = os.path.join(
        out_dir,
        "val_roc_curve.png",
    )
    figure_roc.savefig(
        roc_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    print(f"[保存11] 验证集 ROC 曲线 -> {roc_path}")

    save_resume_progress(
        out_dir=out_dir,
        all_metrics=all_metrics,
        experiment_repeats=experiment_repeats,
        status="complete",
    )

    print("\n" + "=" * 70)
    print(
        f"Final Report: averaged over {number_of_champions} "
        f"CNN champion models (VAL set)"
    )
    print("-" * 70)
    print(f"{'Metric':<14}{'Mean':>8}{'95% CI':>22}")

    for _, row in summary_df.iterrows():
        confidence_interval = (
            f"[{row['CI95_Lower']:.3f}, "
            f"{row['CI95_Upper']:.3f}]"
        )
        print(
            f"{row['Metric']:<14}"
            f"{row['Mean']:>8.3f}"
            f"{confidence_interval:>22}"
        )

    print("-" * 70)
    print_classification_result(combined_cm)
    print("=" * 70)
    print(
        "\n注意: 以上是 CNN 验证集结果，用于筛选模型和随机种子，"
        "不是最终测试集结果。"
    )

    return {
        "summary_df": summary_df,
        "per_repeat_df": per_repeat_df,
        "seed_summary_df": seed_summary_df,
        "selected_panel_df": selected_panel_df,
        "mean_auc": mean_auc,
        "std_auc": std_auc,
        "original_mlp_parameter_count": (
            original_mlp_parameter_count
        ),
        "cnn_parameter_count": cnn_parameter_count,
        "loaded_checkpoint_count_at_start": len(existing_indices),
    }

In [ ]:
cnn_formal_result = experiment(
    './data/voc_dataset_1+2_vs_3_with_unknown.mat',
    './result/unknown_val_cnn_seed_selection/all_masks.pt',
    num_cluster=3,
    experiment_repeats=100,
    repeats=20,
    epochs=300,
    sparsity_lambda=1e-2,
    temp_start=1.0,
    temp_end=0.3,
    seed=42,
    panel_threshold=0.5,
    cnn_channels=(16, 32),
    cnn_pooled_length=8,
    cnn_dropout=0.30,
    resume=True,
)